In [ ]:
%pip install pandas


In [ ]:
import pandas as pd

print("1. Membaca data Red Team...")
# Redteam format: time, source_user, source_computer, dest_computer
redteam_cols = ['time', 'src_user', 'src_comp', 'dst_comp']
redteam_df = pd.read_csv('../Dataset RM/Pick/redteam.txt.gz', names=redteam_cols)

# Trik Optimasi: Ubah red team menjadi struktur 'Set' agar proses pencarian sangat cepat
# Kita gabungkan kombinasi (waktu, source_computer, dest_computer)
attack_set = set(zip(redteam_df['time'], redteam_df['src_comp'], redteam_df['dst_comp']))
print(f"-> Ditemukan {len(attack_set)} log serangan Red Team.")

In [ ]:
print("\n2. Membaca sampel Auth Logs...")
# Auth format: time, src_user, dst_user, src_comp, dst_comp, auth_type, logon_type, orientation, success
auth_cols = ['time', 'src_user', 'dst_user', 'src_comp', 'dst_comp', 'auth_type', 'logon_type', 'orientation', 'success']

# PENTING: Kita baca 500.000 baris pertama dulu agar RAM tidak jebol saat ujicoba
auth_df = pd.read_csv('auth.txt.gz', names=auth_cols, nrows=500000)
print(f"-> Berhasil membaca {len(auth_df)} baris log autentikasi.")

In [ ]:
print("\n3. Proses Pemetaan Label (Mencari Anomali)...")
# Fungsi untuk mengecek apakah sebuah baris log adalah serangan
def check_attack(row):
    if (row['time'], row['src_comp'], row['dst_comp']) in attack_set:
        return 1
    return 0

# Aplikasikan fungsi ke dataframe untuk membuat kolom baru 'is_attack'
auth_df['is_attack'] = auth_df.apply(check_attack, axis=1)

# Hitung hasilnya
jumlah_serangan = auth_df['is_attack'].sum()
print(f"-> Selesai! Ditemukan {jumlah_serangan} serangan dari {len(auth_df)} log.")

In [ ]:
print("\n4. Menyimpan hasil ke format CSV bersih...")
# Simpan hasil sampel ini ke file baru agar tidak perlu mengulang proses lama ini
output_file = 'lognet_labeled_sample.csv'
auth_df.to_csv(output_file, index=False)
print(f"-> File {output_file} berhasil dibuat dan siap diubah menjadi graf!")